# 06 - Supply And Mint-Control Analysis

This notebook evaluates SUMR supply state (total supply vs cap), with optional live RPC refresh.

**Source classes:** ONCHAIN_EXECUTED artifact + optional live chain reads.


## Scope

1. Load latest tracked supply snapshot from evidence artifacts.
2. Compute minted share and remaining mintable capacity.
3. Optionally compare with live on-chain values.
4. If available, inspect local unlock schedule artifacts.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.config import BASE_RPC_URL, SUMR_TOKEN
from web3 import Web3

RUN_LIVE_SUPPLY_READ = False
EVIDENCE_DIR = nbx.latest_evidence_dir()
if EVIDENCE_DIR is None:
    raise FileNotFoundError("No evidence directory found.")

supply_path = EVIDENCE_DIR / "sumr_supply_snapshot.json"
supply = nbx.load_json(supply_path)

base_total = float(supply.get("totalSupply_tokens") or 0.0)
base_cap = float(supply.get("cap_tokens") or 0.0)
remaining = max(base_cap - base_total, 0.0)

snapshot_df = pd.DataFrame([
    {
        "source": "evidence_snapshot",
        "retrieved_utc": supply.get("retrieved_utc"),
        "total_supply_tokens": base_total,
        "cap_tokens": base_cap,
        "remaining_mintable_tokens": remaining,
        "minted_share": (base_total / base_cap) if base_cap > 0 else np.nan,
    }
])
display(snapshot_df)


In [ ]:
live_df = pd.DataFrame()
if RUN_LIVE_SUPPLY_READ:
    if not BASE_RPC_URL:
        raise RuntimeError("BASE_RPC_URL is required for live reads.")

    w3 = Web3(Web3.HTTPProvider(BASE_RPC_URL, request_kwargs={"timeout": 30}))
    if not w3.is_connected():
        raise ConnectionError("Could not connect to Base RPC endpoint.")

    abi = [
        {"name": "totalSupply", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "uint256"}]},
        {"name": "cap", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "uint256"}]},
        {"name": "decimals", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "uint8"}]},
    ]
    contract = w3.eth.contract(address=Web3.to_checksum_address(SUMR_TOKEN), abi=abi)
    decimals = int(contract.functions.decimals().call())
    total_raw = int(contract.functions.totalSupply().call())
    cap_raw = int(contract.functions.cap().call())

    total = total_raw / (10 ** decimals)
    cap = cap_raw / (10 ** decimals)

    live_df = pd.DataFrame([
        {
            "source": "live_chain",
            "retrieved_utc": nbx.utc_now_iso(),
            "total_supply_tokens": total,
            "cap_tokens": cap,
            "remaining_mintable_tokens": max(cap - total, 0.0),
            "minted_share": (total / cap) if cap > 0 else np.nan,
            "block_number": int(w3.eth.block_number),
        }
    ])

if not live_df.empty:
    display(live_df)
else:
    display(Markdown("Live supply read skipped (`RUN_LIVE_SUPPLY_READ=False`)."))


In [ ]:
row = snapshot_df.iloc[0]
minted = float(row["total_supply_tokens"])
remaining = float(row["remaining_mintable_tokens"])

fig, ax = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
ax.pie(
    [minted, remaining],
    labels=["Minted", "Remaining to cap"],
    autopct="%1.2f%%",
    colors=["#2a9d8f", "#e9c46a"],
    startangle=90,
)
ax.set_title("SUMR Supply State (Snapshot)")
plt.show()


In [ ]:
unlock_path = ROOT / "results" / "tables" / "investor_unlock_schedule_next_24m_latest.csv"
if unlock_path.exists():
    unlock_df = pd.read_csv(unlock_path)
    display(unlock_df.head(12))

    date_col = "date" if "date" in unlock_df.columns else unlock_df.columns[0]
    candidate_cols = [
        "monthly_newly_liquid_sumr",
        "monthly_newly_liquid_SUMR",
        "monthly_newly_liquid_tokens",
        "monthly_newly_liquid",
    ]
    value_col = next((c for c in candidate_cols if c in unlock_df.columns), None)

    if value_col is None:
        numeric_cols = [
            c for c in unlock_df.columns
            if c != date_col and pd.api.types.is_numeric_dtype(unlock_df[c])
        ]
        value_col = numeric_cols[0] if numeric_cols else None

    if value_col is None:
        display(Markdown(
            "Unlock CSV found, but no numeric unlock column was detected. "
            f"Available columns: `{list(unlock_df.columns)}`"
        ))
    else:
        fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
        ax.bar(unlock_df[date_col].astype(str), unlock_df[value_col], color="#457b9d")
        ax.set_title("Modeled Monthly Newly-Liquid SUMR")
        ax.set_ylabel("SUMR")
        ax.tick_params(axis="x", rotation=45)
        plt.show()
else:
    display(Markdown(
        "Unlock schedule CSV is not in the public-tracked set by default. "
        "Generate locally via `make report` if you want unlock-path visuals in this notebook."
    ))
